# SeedVR2 Lite — Colab CLI

Select a GPU runtime (Ubuntu 24.04, Python 3.13, PyTorch 2.11.0+cu128, CUDA toolkit 12.8). Run the cells in order. Weights, outputs and the setup diagnostic report go to Google Drive; code stays on Colab's local disk.

Edit `DRIVE_ROOT`, optional memory allowances, and `INSTALL_FLASH_ATTN` in the paths cell. The default PyTorch SDPA attention does not need FlashAttention. The optional prebuilt wheel requires a compatible Python/PyTorch/CUDA build and an Ampere/Ada/Hopper GPU (SM 8.x/9.x); T4 cannot use it. Each installation step stops on failure. FFmpeg and ffprobe must actually report version 9 and provide x264/x265 encoders.


In [ ]:
import platform
import subprocess
import sys
import torch

print("OS:", platform.platform())
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "PyTorch CUDA:", torch.version.cuda)
if platform.system() != "Linux" or sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select the supported Ubuntu 24.04 / Python 3.13 Colab runtime")
if torch.__version__ != "2.11.0+cu128" or torch.version.cuda != "12.8":
    raise RuntimeError("This notebook requires Colab PyTorch 2.11.0+cu128 (CUDA 12.8)")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab")
props = torch.cuda.get_device_properties(0)
print(f"GPU 0: {props.name}, {props.total_memory / 2**30:.1f} GiB")
subprocess.run(["nvidia-smi"], check=True)

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/delvilo/SeedVR2Lite_VideoUpscaler.git"
REPO_REF = "main"
REPO_DIR = Path("/content/SeedVR2Lite_VideoUpscaler")
DRIVE_ROOT = Path("/content/drive/MyDrive/SeedVR2")
INPUT_DIR = DRIVE_ROOT / "input"
MODEL_DIR = DRIVE_ROOT / "models"
OUTPUT_DIR = DRIVE_ROOT / "output"
RAM_BUDGET_GB = 0.0  # 0: 80% of free RAM; explicit GiB values are capped at available RAM
VRAM_BUDGET_GB = 0.0  # 0: 90% of free VRAM; explicit GiB values are capped at free VRAM
INSTALL_FLASH_ATTN = False  # set True only for FlashAttention 2 on a compatible GPU
for directory in (INPUT_DIR, MODEL_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)

print("Source:", REPO_DIR)
print("Inputs:", INPUT_DIR)
print("Weights:", MODEL_DIR)
print("Results:", OUTPUT_DIR)


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from src.utils.runtime import validate_runtime
from colab.workflow import install_dependencies, write_diagnostics
validate_runtime()

installation = install_dependencies(REPO_DIR, install_flash_attn=INSTALL_FLASH_ATTN)
subprocess.run([sys.executable, str(REPO_DIR / "inference_cli.py"), "--help"], check=True)
DIAGNOSTICS_PATH = write_diagnostics(
    REPO_DIR, INPUT_DIR, MODEL_DIR, OUTPUT_DIR, installation,
    ram_budget_gb=RAM_BUDGET_GB, vram_budget_gb=VRAM_BUDGET_GB,
)
BUDGET_ARGS = ["--ram_budget_gb", str(RAM_BUDGET_GB), "--vram_budget_gb", str(VRAM_BUDGET_GB)]


## Quick sample

This short sample downloads the default DiT and VAE weights to Drive on first use, then processes up to five frames. Keep the model folder for later sessions. The chosen allowances and phase peaks are recorded separately in the setup diagnostic JSON and sample memory JSONL files.


In [ ]:
sample = REPO_DIR / "examples/inputs/Eyes_212x120.mp4"
sample_output = OUTPUT_DIR / "sample_upscaled.mp4"
subprocess.run([
    sys.executable, str(REPO_DIR / "inference_cli.py"), str(sample),
    "--output", str(sample_output),
    "--model_dir", str(MODEL_DIR),
    "--resolution", "240", "--batch_size", "5", "--load_cap", "5",
    "--video_backend", "ffmpeg",
    "--memory_report", str(OUTPUT_DIR / "sample_memory.jsonl"),
    *BUDGET_ARGS,
], check=True)
print("Saved:", sample_output)


## Your video

Upload `video.mp4` to the input folder on Drive or change the path. The output and memory JSONL report are saved in the output folder. `colab_diagnostics.json` records the exact source revision, paths, installed versions, encoders, available RAM/VRAM and effective memory allowances at setup time. Memory planning caps the input window and batch size; VAE tiling and CPU offload are enabled by default. Caught OOMs restart the file with smaller workloads while keeping the requested resolution and seed. Batch/tile changes can alter pixels.


In [ ]:
input_video = INPUT_DIR / "video.mp4"
if not input_video.is_file():
    raise FileNotFoundError(f"Upload a video to {input_video}, or edit input_video in this cell")

result_video = OUTPUT_DIR / f"{input_video.stem}_upscaled.mp4"
subprocess.run([
    sys.executable, str(REPO_DIR / "inference_cli.py"), str(input_video),
    "--output", str(result_video),
    "--model_dir", str(MODEL_DIR),
    "--resolution", "1080", "--batch_size", "5",
    "--chunk_size", "33", "--temporal_overlap", "3",
    "--video_backend", "ffmpeg",
    "--memory_report", str(OUTPUT_DIR / f"{input_video.stem}_memory.jsonl"),
    *BUDGET_ARGS,
], check=True)
print("Saved:", result_video)
